In [1]:
import sempy
import sempy.fabric as fabric
import pandas

StatementMeta(, b7600b4b-e6c2-4826-b98d-9a27d49ec61a, 5, Finished, Available, Finished)

In [2]:
%run 2. Parameters

StatementMeta(, b7600b4b-e6c2-4826-b98d-9a27d49ec61a, 6, Finished, Available, Finished)

In [3]:
from sempy import fabric

# Get the metadata including data types
list_columns_df = fabric.list_relationships(dataset=datasetId, workspace=workspaceId)

# Add columns to your pandas DataFrame
list_columns_df['workspace_id'] = workspaceId
list_columns_df['dataset_id'] = datasetId
list_columns_df['database_name'] = semantic_model_name

# Automatically replace spaces with underscores in all column names
list_columns_df.columns = list_columns_df.columns.str.replace(' ', '_').str.upper()

# Drop columns with void/null datatypes
list_columns_df = list_columns_df.dropna(axis=1, how='all')

# Alternatively, explicitly drop the problematic column
# list_columns_df = list_columns_df.drop(columns=['sort_by_column'], errors='ignore')

# Define table name
table_name = "r_Relationships"

# Drop the table if it exists to clear any metadata issues
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Convert to Spark DataFrame and write
spark_list_columns_df = spark.createDataFrame(list_columns_df)
spark_list_columns_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"Successfully wrote {spark_list_columns_df.count()} rows to table '{table_name}'")

# Now query and display
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1000")
display(df)

StatementMeta(, b7600b4b-e6c2-4826-b98d-9a27d49ec61a, 7, Finished, Available, Finished)

Successfully wrote 6 rows to table 'r_Relationships'


SynapseWidget(Synapse.DataFrame, 5eaf847b-f130-463c-945a-83cbd56e1628)

In [4]:


# =============================================
# ADD SWAPPED ROWS
# =============================================
import pandas as pd

mult_swap = {
    'm:1': '1:m',
    '1:m': 'm:1',
    '1:1': '1:1',
    'm:m': 'm:m'
}

swapped_df = list_columns_df.copy()
swapped_df['FROM_TABLE']   = list_columns_df['TO_TABLE']
swapped_df['FROM_COLUMN']  = list_columns_df['TO_COLUMN']
swapped_df['TO_TABLE']     = list_columns_df['FROM_TABLE']
swapped_df['TO_COLUMN']    = list_columns_df['FROM_COLUMN']
swapped_df['MULTIPLICITY'] = list_columns_df['MULTIPLICITY'].map(
                                lambda x: mult_swap.get(str(x).strip().lower(), x)
                             )

# Combine original + swapped (alternating: original row then its swap)
list_columns_doubled_df = pd.concat([list_columns_df, swapped_df]) \
                            .sort_index(kind='stable') \
                            .reset_index(drop=True)

print(f"Original rows : {len(list_columns_df)}")
print(f"Total rows after doubling: {len(list_columns_doubled_df)}")

# -----------------------------
# Write doubled table
# -----------------------------
table_name_doubled = "r_Relationships_Doubled"

spark.sql(f"DROP TABLE IF EXISTS {table_name_doubled}")

spark_doubled_df = spark.createDataFrame(list_columns_doubled_df)
spark_doubled_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(table_name_doubled)

print(f"Successfully wrote {spark_doubled_df.count()} rows to table '{table_name_doubled}'")

df_doubled = spark.sql(f"SELECT * FROM {table_name_doubled} LIMIT 1000")
display(df_doubled)

StatementMeta(, b7600b4b-e6c2-4826-b98d-9a27d49ec61a, 8, Finished, Available, Finished)

Original rows : 6
Total rows after doubling: 12
Successfully wrote 12 rows to table 'r_Relationships_Doubled'


SynapseWidget(Synapse.DataFrame, 72a311d8-282f-4964-b946-d7c9136c7c0b)

In [6]:
import pandas as pd
from datetime import datetime
from pyspark.sql.types import StringType, StructField, StructType

# ✅ Use the pandas df directly instead of display()
pandas_df = list_columns_doubled_df.copy()

# Convert all column names to uppercase
pandas_df.columns = pandas_df.columns.str.upper()

# Add timestamp column
pandas_df["DATABASE_TIMESTAMP"] = f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"

# Determine version number based on existing data
table_name = "Relationships"

if spark.catalog.tableExists(table_name):
    existing_df = spark.read.table(table_name)
    
    max_version_row = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()
    
    max_version = max_version_row[0]['max_ver'] if max_version_row else None
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
pandas_df["DATABASE_VERSION"] = f"{semantic_model_name} | V{next_version}"

# ── Fix void/null-only columns before converting to Spark ──────────────────
for col in pandas_df.columns:
    if pandas_df[col].dtype == object or str(pandas_df[col].dtype) == 'void':
        pandas_df[col] = pandas_df[col].astype(str).replace('None', None).replace('nan', None)

# Build an explicit schema so Spark doesn't have to guess
schema = StructType([
    StructField(col, StringType(), True) for col in pandas_df.columns
])

# Convert back to Spark only for the write step — using explicit schema
spark_df = spark.createDataFrame(pandas_df, schema=schema)
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

print(f"Successfully wrote to table '{table_name}' with version V{next_version}")
display(pandas_df)

StatementMeta(, b7600b4b-e6c2-4826-b98d-9a27d49ec61a, 10, Finished, Available, Finished)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Exception thrown when converting pandas.Series (bool) with name 'ACTIVE' to Arrow Array (string).
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


Successfully wrote to table 'Relationships' with version V1


SynapseWidget(Synapse.DataFrame, a31b73f0-35d8-4391-b586-bdd918cc3ecc)